## Load the summary information and preprocessed data

In [1]:
import os
import json
import pandas as pd

from pathlib import Path

from single_sample_gsea import ss_gsea


# Get GEO_ID from folder name (same pattern as prepare_data.ipynb)
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))


description_path = Path('description.json')
preprocessed_csv_path = Path(f"{GEO_ID}_preprocessed.csv")

if not description_path.exists():
    raise FileNotFoundError(
        "description.json not found. Please run 'prepare_data.ipynb' first to generate it."
    )

if not preprocessed_csv_path.exists():
    raise FileNotFoundError(
        f"{preprocessed_csv_path} not found. Please run 'preprocess_data.ipynb' first to generate it."
    )

with open(description_path) as f:
    summary_data = json.load(f)

df = pd.read_csv(preprocessed_csv_path)

print("Data contains", df.shape[0], "samples and", df.shape[1], "columns.")
print("First few rows of the data:")
df.head()


Data contains 96 samples and 7778 columns.
First few rows of the data:


,dead or last follow up (days),dead,ID,age at diagnosis,clinical stage,MYCN amplification,17q unbalance,patient number,probe ID tumor,RNA-seq data,...,COX2,ATP8,ATP6,COX3,ND3,ND4L,ND4,ND5,ND6,CYTB
0,2121.0,0.0,1,356,1,-,+,NaN,984760.0,+,...,8.361404,7.840448,8.372249,8.109869,6.421469,6.893357,9.263799,8.778961,7.866203,8.322789
1,1456.0,0.0,2,1197,1,-,-,NaN,1178899.0,+,...,9.172643,7.889168,9.005146,9.100367,7.104996,7.129575,9.759173,9.271679,9.520435,8.990730
2,590.0,0.0,4,298,1,-,-,NaN,2217756.0,+,...,7.959898,6.805186,8.079660,8.220222,6.292021,6.856885,9.079503,8.695052,7.957350,8.038179
3,1631.0,0.0,5,1852,1,-,-,NaN,2238721.0,+,...,7.889291,6.553900,8.233543,7.954751,6.139533,6.657749,9.003603,8.600086,7.712642,8.325577
4,54.0,0.0,6,95,1,-,-,NaN,2238819.0,+,...,9.119654,7.903819,9.356463,9.497230,7.502189,7.892376,10.438208,9.567538,8.763026,9.608265


## Load gene sets

In [2]:
MSIGDB_JSON = Path("../../resources/h.all.v2026.1.Hs.json")

with MSIGDB_JSON.open(encoding="utf-8") as f:
    gene_sets = {name: data["geneSymbols"] for name, data in json.load(f).items()}

print(f"Loaded {len(gene_sets)} gene sets.")


Loaded 50 gene sets.


## Compute single sample GSEA on the preprocessed data

Select only gene columns for the computation.

In [3]:
# IMPORTANT: This is dataset-specific and is the only variable that needs to be changed for each of the dataset.
sample_id_column = 'sample_title'

gene_expression_df = df.iloc[:, summary_data['data_summary']['clinical_features']:].copy()
gene_expression_df.index = df[sample_id_column].astype(str)
gene_expression_df = gene_expression_df.T
gene_expression_df.index.name = "gene"

result = ss_gsea(gene_expression_df, gene_sets)


## Merge the results with the clinical data

The output table should have the same number of rows as before and columns should sum up to clinical data + 50 gene sets.

In [4]:
clinical_df = df.iloc[:, :summary_data["data_summary"]["clinical_features"]].copy()
clinical_df = clinical_df.set_index(sample_id_column)

output_df = pd.concat([clinical_df, result], axis=1)
output_df = output_df.reset_index(drop=False)

output_df.to_csv(f"{GEO_ID}_preprocessed_ssgsea.csv", index=False)

# sanity check (there is 50 gene sets)
assert output_df.shape[1] == summary_data["data_summary"]["clinical_features"] + 50
